# 📊 Linear Discriminant Analysis (LDA)
### Supervised Dimensionality Reduction — Maximizing Class Separability

---

**What you'll learn in this notebook:**
- Why LDA is different from PCA (supervised vs unsupervised)
- How LDA finds the best axis to separate classes
- Apply LDA on the classic Iris dataset
- Visualize 1D and 2D projections
- Compare class separation: Before and After LDA
- Understand explained variance in LDA components
- Compare LDA vs PCA on the same dataset

**Dataset**: Iris (150 samples, 4 features, 3 classes)

> 💡 **Tool**: scikit-learn's `LinearDiscriminantAnalysis` + matplotlib for visualization

---

## 📦 Step 0: Import Libraries

In [ ]:
# Core libraries
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings('ignore')

# Scikit-learn tools
from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.decomposition import PCA
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import Pipeline

# Set style
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 10
np.random.seed(42)

print("All libraries imported successfully ✅")

---
## 📂 Step 1: Load the Iris Dataset

### Why Iris?
The Iris dataset is perfect for learning LDA because:
- 3 classes (Setosa, Versicolor, Virginica) — so LDA can produce up to **2 components** (n_classes − 1)
- 4 features (sepal length, sepal width, petal length, petal width)
- One class is linearly separable, two others are not — interesting challenge!
- Small and fast to train

**Goal**: Reduce 4 features → 2 LDA components, and see if classes are clearly separated.

In [ ]:
# Load the Iris dataset
iris = load_iris()

X = iris.data          # Shape: (150, 4) — 150 samples, 4 features
y = iris.target        # Shape: (150,)   — class labels: 0, 1, 2
feature_names  = iris.feature_names
class_names    = iris.target_names   # ['setosa', 'versicolor', 'virginica']

# Color palette for 3 classes
COLORS = ['#e74c3c', '#2ecc71', '#3498db']  # red, green, blue

print(f"Dataset shape  : {X.shape}")
print(f"Features       : {feature_names}")
print(f"Classes        : {class_names}")
print(f"Samples/class  : {np.bincount(y)}")
print(f"\nFirst 5 rows:\n{X[:5]}")

In [ ]:
# Visualize raw feature distributions (before LDA)
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.flatten()

for feat_idx, ax in enumerate(axes):
    for cls_idx, cls_name in enumerate(class_names):
        # Plot histogram for each class
        ax.hist(X[y == cls_idx, feat_idx], bins=15, alpha=0.6,
                color=COLORS[cls_idx], label=cls_name, edgecolor='white')
    ax.set_title(f'Feature: {feature_names[feat_idx]}', fontweight='bold')
    ax.set_xlabel('Value (cm)')
    ax.set_ylabel('Count')
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

plt.suptitle('Raw Feature Distributions by Class (Before LDA)', 
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()
print("\n💡 Observation: Some features already separate Setosa well, but")
print("   Versicolor and Virginica overlap in many features individually.")

---
## ⚖️ Step 2: Preprocess — Scale the Features

### Why scale before LDA?
LDA computes scatter matrices using distances and spreads. If one feature is in centimeters and another in millimeters, the larger-scaled feature dominates unfairly.

**StandardScaler** transforms each feature to have:
- Mean = 0
- Standard Deviation = 1

This ensures all features contribute equally to the scatter matrices.

> ⚠️ Always fit the scaler on **training data only** and use it to transform test data. Here we use the full dataset for demonstration.

In [ ]:
# Scale features to zero mean and unit variance
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Verify scaling
print("Before Scaling:")
print(f"  Mean per feature : {X.mean(axis=0).round(2)}")
print(f"  Std  per feature : {X.std(axis=0).round(2)}")

print("\nAfter Scaling:")
print(f"  Mean per feature : {X_scaled.mean(axis=0).round(5)}  ← approx 0")
print(f"  Std  per feature : {X_scaled.std(axis=0).round(5)}   ← approx 1")

---
## 🔬 Step 3: Apply LDA — Reduce to 2 Components

### How LDA decides the number of components
- **Maximum components** = min(n_classes − 1, n_features)
- For Iris: min(3 − 1, 4) = **2 components**

So LDA can perfectly compress 4D Iris data into 2D — and these 2 dimensions are **specifically chosen to best separate the 3 classes**.

```python
lda = LinearDiscriminantAnalysis(n_components=2)
X_lda = lda.fit_transform(X_scaled, y)   # y (labels) is required!
```

> Key difference from PCA: `fit_transform(X, y)` — the `y` argument is what makes this supervised.

In [ ]:
# Apply LDA with 2 components (maximum for 3 classes)
lda = LinearDiscriminantAnalysis(n_components=2)

# fit_transform: learns the discriminant axes AND projects data
# Note: y (class labels) is REQUIRED — this is what makes LDA supervised
X_lda = lda.fit_transform(X_scaled, y)

print(f"Original data shape : {X_scaled.shape}  (150 samples × 4 features)")
print(f"LDA projected shape : {X_lda.shape}   (150 samples × 2 components)")
print(f"\nExplained variance ratio:")
for i, ratio in enumerate(lda.explained_variance_ratio_):
    print(f"  LD{i+1}: {ratio:.4f}  ({ratio*100:.1f}%)")
print(f"  Total: {lda.explained_variance_ratio_.sum():.4f}  ({lda.explained_variance_ratio_.sum()*100:.1f}%)")

---
## 📊 Step 4: Visualize the LDA Projection (2D)

We now plot the 150 samples in the 2D LDA space.

**What to look for:**
- Each color = one class
- Well-separated clusters = LDA is working
- Overlapping clusters = those classes are harder to separate even with LDA

Recall: Setosa is known to be easily separable; Versicolor and Virginica are the hard pair.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))

for cls_idx, cls_name in enumerate(class_names):
    mask = y == cls_idx
    ax.scatter(
        X_lda[mask, 0], X_lda[mask, 1],
        c=COLORS[cls_idx], label=cls_name,
        alpha=0.8, edgecolors='white', linewidths=0.5, s=80
    )
    
    # Mark the class centroid
    cx, cy = X_lda[mask, 0].mean(), X_lda[mask, 1].mean()
    ax.scatter(cx, cy, c=COLORS[cls_idx], marker='*', s=300,
               edgecolors='black', linewidths=1.2, zorder=5)
    ax.annotate(f'{cls_name}\ncenter', (cx, cy),
                textcoords='offset points', xytext=(10, 8),
                fontsize=8, fontweight='bold', color=COLORS[cls_idx])

ax.set_xlabel(f'LD1 ({lda.explained_variance_ratio_[0]*100:.1f}% variance)', fontsize=11)
ax.set_ylabel(f'LD2 ({lda.explained_variance_ratio_[1]*100:.1f}% variance)', fontsize=11)
ax.set_title('LDA Projection — Iris Dataset (4D → 2D)', fontsize=13, fontweight='bold')
ax.legend(fontsize=10, framealpha=0.9)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("\n💡 Observations:")
print("   • Setosa (red) is perfectly separated from the other two")
print("   • Versicolor and Virginica have slight overlap — they are naturally similar")
print("   • LD1 carries most of the separating power")

---
## 📉 Step 5: Visualize LDA on 1 Component (1D Projection)

LDA with just **1 component** (LD1) already captures the most important separation direction. A 1D histogram shows how well LD1 alone separates the classes.

This is useful when you want the most compact possible representation — just a single number — that still distinguishes classes.

In [ ]:
# Apply LDA with just 1 component
lda_1d = LinearDiscriminantAnalysis(n_components=1)
X_lda_1d = lda_1d.fit_transform(X_scaled, y)  # Shape: (150, 1)

# Plot 1D histograms for each class
fig, ax = plt.subplots(figsize=(10, 4))

for cls_idx, cls_name in enumerate(class_names):
    mask = y == cls_idx
    ax.hist(X_lda_1d[mask, 0], bins=20, alpha=0.65,
            color=COLORS[cls_idx], label=cls_name, edgecolor='white')
    
    # Mark the mean
    mean_val = X_lda_1d[mask, 0].mean()
    ax.axvline(mean_val, color=COLORS[cls_idx], linewidth=2.5,
               linestyle='--', alpha=0.9)

ax.set_xlabel('LD1 Value (Single Linear Discriminant)', fontsize=11)
ax.set_ylabel('Count', fontsize=11)
ax.set_title('LDA — 1D Projection (LD1 only)', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\n💡 Even with just 1 number (LD1), the three classes are well separated!")
print(f"   Class means on LD1: Setosa={X_lda_1d[y==0].mean():.2f}, "
      f"Versicolor={X_lda_1d[y==1].mean():.2f}, Virginica={X_lda_1d[y==2].mean():.2f}")

---
## 🔍 Step 6: Inspect the LDA Components

The LDA **scalings** (also called coefficients or loadings) tell us how much each original feature contributes to each Linear Discriminant.

**Interpreting scalings:**
- Large absolute value → that feature has a **strong influence** on this discriminant
- Positive vs negative → direction of influence

This is useful for understanding **which features drive class separation**.

In [ ]:
# Get the LDA coefficients (scalings)
# Shape: (n_features, n_components) = (4, 2)
scalings = lda.scalings_

print("LDA Component Scalings (how much each feature contributes):")
print(f"{'Feature':<30} {'LD1':>8} {'LD2':>8}")
print("-" * 50)
for fname, ld1, ld2 in zip(feature_names, scalings[:, 0], scalings[:, 1]):
    print(f"{fname:<30} {ld1:>8.3f} {ld2:>8.3f}")

# Visualize as bar chart
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

short_names = ['Sepal L', 'Sepal W', 'Petal L', 'Petal W']

for i, ax in enumerate(axes):
    bars = ax.bar(short_names, scalings[:, i],
                  color=['#e74c3c' if v >= 0 else '#3498db' for v in scalings[:, i]],
                  edgecolor='white', linewidth=0.8)
    ax.axhline(0, color='black', linewidth=0.8)
    ax.set_title(f'LD{i+1} — Feature Contributions', fontweight='bold')
    ax.set_ylabel('Scaling Coefficient')
    ax.grid(axis='y', alpha=0.3)
    
    # Add value labels on bars
    for bar, val in zip(bars, scalings[:, i]):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05 * np.sign(val),
                f'{val:.2f}', ha='center', va='bottom' if val >= 0 else 'top', fontsize=9)

plt.suptitle('Feature Importance in LDA Components', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## ⚖️ Step 7: Compare LDA vs PCA — Side by Side

### The Core Difference
- **PCA**: Finds directions of **maximum variance** — ignores class labels
- **LDA**: Finds directions of **maximum class separation** — uses class labels

For labeled data, LDA often gives much better class separation in fewer dimensions.

**Analogy:**  
PCA is like spreading items on a table to see as much of them as possible.  
LDA is like sorting items by type and spreading *each type* apart from the others.

In [ ]:
# Apply PCA with 2 components for comparison
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)   # No y — unsupervised!

# Side-by-side comparison plot
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

datasets = [
    (X_pca, pca.explained_variance_ratio_, 'PC', 'PCA (Unsupervised)'),
    (X_lda, lda.explained_variance_ratio_, 'LD', 'LDA (Supervised)')
]

for ax, (X_proj, var_ratio, prefix, title) in zip(axes, datasets):
    for cls_idx, cls_name in enumerate(class_names):
        mask = y == cls_idx
        ax.scatter(
            X_proj[mask, 0], X_proj[mask, 1],
            c=COLORS[cls_idx], label=cls_name,
            alpha=0.8, edgecolors='white', linewidths=0.5, s=70
        )
    
    ax.set_xlabel(f'{prefix}1 ({var_ratio[0]*100:.1f}% variance)', fontsize=11)
    ax.set_ylabel(f'{prefix}2 ({var_ratio[1]*100:.1f}% variance)', fontsize=11)
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.legend(fontsize=9, framealpha=0.9)
    ax.grid(alpha=0.3)

plt.suptitle('LDA vs PCA — 2D Projection of Iris Dataset', 
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n💡 Observations:")
print("   • PCA separates classes reasonably but wasn't designed to")
print("   • LDA gives cleaner, wider separation between all 3 classes")
print("   • LDA's advantage: it uses class label information during projection")

---
## 📏 Step 8: Quantify Class Separation — Fisher Score

We can measure how well-separated the classes are using a simple metric: the **ratio of between-class variance to within-class variance** on the projected data.

A higher ratio = better separation. Let's compare LDA vs PCA on this metric.

In [ ]:
def fisher_score(X_proj, y, labels):
    """
    Compute a separation score: ratio of between-class to within-class variance.
    Higher = better class separation.
    Only uses the first component (1D) for simplicity.
    """
    overall_mean = X_proj[:, 0].mean()
    
    # Between-class variance: how far are class means from overall mean
    between = sum(
        (y == c).sum() * (X_proj[y == c, 0].mean() - overall_mean) ** 2
        for c in labels
    ) / len(y)
    
    # Within-class variance: spread inside each class
    within = sum(
        X_proj[y == c, 0].var() for c in labels
    ) / len(labels)
    
    return between / (within + 1e-10)  # Avoid division by zero

labels = np.unique(y)
lda_score = fisher_score(X_lda, y, labels)
pca_score = fisher_score(X_pca, y, labels)

print("Class Separation Score (Between/Within Variance Ratio — higher is better):")
print(f"  LDA (LD1) : {lda_score:.2f}")
print(f"  PCA (PC1) : {pca_score:.2f}")
print(f"  LDA is {lda_score/pca_score:.1f}x better at class separation than PCA")

# Bar chart
fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(['PCA (PC1)', 'LDA (LD1)'], [pca_score, lda_score],
              color=['#95a5a6', '#2ecc71'], edgecolor='white', width=0.5)
for bar, val in zip(bars, [pca_score, lda_score]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f'{val:.2f}', ha='center', fontweight='bold', fontsize=12)
ax.set_ylabel('Separation Score (↑ better)', fontsize=11)
ax.set_title('Class Separation: LDA vs PCA', fontsize=12, fontweight='bold')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

---
## 🎯 Step 9: LDA as a Classifier

LDA isn't just for dimensionality reduction — it can also **directly classify** new samples!

After finding the discriminant axes, LDA assigns each sample to the class whose **centroid (mean)** is closest in the projected space (using Mahalanobis distance).

Let's evaluate LDA as a classifier using cross-validation.

In [ ]:
# Evaluate LDA as a classifier using 5-fold cross-validation
# Pipeline: scale first, then LDA classify
lda_clf_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('lda', LinearDiscriminantAnalysis())  # No n_components — full classification mode
])

cv_scores = cross_val_score(lda_clf_pipeline, X, y, cv=5, scoring='accuracy')

print("LDA Classifier — 5-Fold Cross-Validation Results:")
print(f"  Accuracy per fold : {[f'{s:.3f}' for s in cv_scores]}")
print(f"  Mean Accuracy     : {cv_scores.mean():.4f} ({cv_scores.mean()*100:.1f}%)")
print(f"  Std Deviation     : ±{cv_scores.std():.4f}")
print()
print("💡 LDA often achieves very high accuracy on well-structured linear problems.")
print("   On Iris, ~98% accuracy is expected — Versicolor/Virginica overlap causes some error.")

# Visualize cross-validation results
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(range(1, 6), cv_scores * 100, color='#3498db', edgecolor='white', alpha=0.85)
ax.axhline(cv_scores.mean() * 100, color='#e74c3c', linewidth=2.5,
           linestyle='--', label=f'Mean: {cv_scores.mean()*100:.1f}%')
ax.set_xlabel('Fold', fontsize=11)
ax.set_ylabel('Accuracy (%)', fontsize=11)
ax.set_title('LDA Classifier — Cross-Validation Accuracy', fontsize=12, fontweight='bold')
ax.set_ylim(90, 102)
ax.set_xticks(range(1, 6))
ax.legend(fontsize=10)
ax.grid(axis='y', alpha=0.3)
for i, score in enumerate(cv_scores):
    ax.text(i + 1, score * 100 + 0.2, f'{score*100:.1f}%', ha='center', fontsize=9)
plt.tight_layout()
plt.show()

---
## 🌐 Step 10: Visualize LDA Decision Regions

In the 2D LDA space, we can visualize where LDA draws its **decision boundaries** — the regions it assigns to each class.

We train a new LDA classifier *in the 2D space* for visualization purposes.

In [ ]:
# Train an LDA classifier in the 2D projected space (for decision boundary visualization)
lda_2d_clf = LinearDiscriminantAnalysis()
lda_2d_clf.fit(X_lda, y)  # Train on 2D LDA features

# Create a meshgrid to fill the 2D space
x_min, x_max = X_lda[:, 0].min() - 1, X_lda[:, 0].max() + 1
y_min, y_max = X_lda[:, 1].min() - 1, X_lda[:, 1].max() + 1
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 400),
                     np.linspace(y_min, y_max, 400))

# Predict class for every point in the grid
Z = lda_2d_clf.predict(np.c_[xx.ravel(), yy.ravel()])
Z = Z.reshape(xx.shape)

# Plot decision regions + actual data points
fig, ax = plt.subplots(figsize=(9, 6))

# Background color regions
background_colors = ['#fadbd8', '#d5f5e3', '#d6eaf8']  # light red, green, blue
from matplotlib.colors import ListedColormap
cmap_bg = ListedColormap(background_colors)
ax.contourf(xx, yy, Z, alpha=0.4, cmap=cmap_bg)
ax.contour(xx, yy, Z, colors='black', linewidths=0.8, linestyles='--', alpha=0.4)

# Scatter actual data
for cls_idx, cls_name in enumerate(class_names):
    mask = y == cls_idx
    ax.scatter(X_lda[mask, 0], X_lda[mask, 1],
               c=COLORS[cls_idx], label=cls_name,
               edgecolors='white', linewidths=0.5, s=80, alpha=0.9)

ax.set_xlabel(f'LD1 ({lda.explained_variance_ratio_[0]*100:.1f}% variance)', fontsize=11)
ax.set_ylabel(f'LD2 ({lda.explained_variance_ratio_[1]*100:.1f}% variance)', fontsize=11)
ax.set_title('LDA Decision Regions in 2D Space', fontsize=13, fontweight='bold')
ax.legend(fontsize=10, framealpha=0.9)
ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()

print("\n💡 The dashed lines are decision boundaries.")
print("   Points in the red region → predicted Setosa")
print("   Points in the green region → predicted Versicolor")
print("   Points in the blue region → predicted Virginica")

---
## 📋 Step 11: Explained Variance — How Much Information Is Retained?

In LDA, each component captures some percentage of the **discriminant information** (ability to separate classes). Let's visualize this and compare to PCA's explained variance.

In [ ]:
# PCA explained variance for all 4 components
pca_full = PCA(n_components=4)
pca_full.fit(X_scaled)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# ── LDA Explained Variance ──
lda_ev = lda.explained_variance_ratio_ * 100
lda_cum = np.cumsum(lda_ev)

axes[0].bar(['LD1', 'LD2'], lda_ev, color=['#2ecc71', '#27ae60'],
            edgecolor='white', alpha=0.85)
ax2 = axes[0].twinx()
ax2.plot(['LD1', 'LD2'], lda_cum, 'ro--', linewidth=2, markersize=8, label='Cumulative')
ax2.set_ylim(0, 110)
ax2.set_ylabel('Cumulative (%)', color='red', fontsize=10)
for i, (ev, label) in enumerate(zip(lda_ev, ['LD1', 'LD2'])):
    axes[0].text(i, ev + 0.5, f'{ev:.1f}%', ha='center', fontweight='bold', fontsize=11)
axes[0].set_title('LDA — Discriminant Variance', fontweight='bold', fontsize=12)
axes[0].set_ylabel('Explained Variance (%)')
axes[0].set_ylim(0, 110)
axes[0].grid(axis='y', alpha=0.3)

# ── PCA Explained Variance ──
pca_ev = pca_full.explained_variance_ratio_ * 100
pca_cum = np.cumsum(pca_ev)
pc_labels = [f'PC{i+1}' for i in range(4)]

axes[1].bar(pc_labels, pca_ev, color=['#3498db', '#2980b9', '#5dade2', '#85c1e9'],
            edgecolor='white', alpha=0.85)
ax3 = axes[1].twinx()
ax3.plot(pc_labels, pca_cum, 'ro--', linewidth=2, markersize=8, label='Cumulative')
ax3.set_ylim(0, 110)
ax3.set_ylabel('Cumulative (%)', color='red', fontsize=10)
for i, ev in enumerate(pca_ev):
    axes[1].text(i, ev + 0.5, f'{ev:.1f}%', ha='center', fontweight='bold', fontsize=9)
axes[1].set_title('PCA — Variance Explained', fontweight='bold', fontsize=12)
axes[1].set_ylabel('Explained Variance (%)')
axes[1].set_ylim(0, 110)
axes[1].grid(axis='y', alpha=0.3)

plt.suptitle('LDA vs PCA — Variance Explained by Components', 
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"LDA: LD1+LD2 captures {lda_cum[-1]:.1f}% of discriminant information (2 components)")
print(f"PCA: PC1+PC2 captures {pca_cum[1]:.1f}% of total variance (2 components)")

---
## 🗂️ Step 12: Summary Dashboard — Everything in One View

In [ ]:
# ── SUMMARY DASHBOARD ──
fig = plt.figure(figsize=(16, 10))
fig.suptitle('LDA Complete Summary — Iris Dataset', fontsize=15, fontweight='bold', y=0.98)

# Panel 1: LDA 2D scatter
ax1 = fig.add_subplot(2, 3, 1)
for cls_idx, cls_name in enumerate(class_names):
    mask = y == cls_idx
    ax1.scatter(X_lda[mask, 0], X_lda[mask, 1], c=COLORS[cls_idx],
                label=cls_name, alpha=0.8, s=40, edgecolors='white', linewidths=0.4)
ax1.set_title('LDA 2D Projection', fontweight='bold')
ax1.set_xlabel(f'LD1 ({lda.explained_variance_ratio_[0]*100:.1f}%)')
ax1.set_ylabel(f'LD2 ({lda.explained_variance_ratio_[1]*100:.1f}%)')
ax1.legend(fontsize=7)
ax1.grid(alpha=0.3)

# Panel 2: PCA 2D scatter
ax2 = fig.add_subplot(2, 3, 2)
for cls_idx, cls_name in enumerate(class_names):
    mask = y == cls_idx
    ax2.scatter(X_pca[mask, 0], X_pca[mask, 1], c=COLORS[cls_idx],
                label=cls_name, alpha=0.8, s=40, edgecolors='white', linewidths=0.4)
ax2.set_title('PCA 2D Projection', fontweight='bold')
ax2.set_xlabel(f'PC1 ({pca_full.explained_variance_ratio_[0]*100:.1f}%)')
ax2.set_ylabel(f'PC2 ({pca_full.explained_variance_ratio_[1]*100:.1f}%)')
ax2.legend(fontsize=7)
ax2.grid(alpha=0.3)

# Panel 3: LDA 1D histogram
ax3 = fig.add_subplot(2, 3, 3)
for cls_idx, cls_name in enumerate(class_names):
    mask = y == cls_idx
    ax3.hist(X_lda_1d[mask, 0], bins=15, alpha=0.6,
             color=COLORS[cls_idx], label=cls_name, edgecolor='white')
ax3.set_title('LDA 1D (LD1 only)', fontweight='bold')
ax3.set_xlabel('LD1 Value')
ax3.set_ylabel('Count')
ax3.legend(fontsize=7)
ax3.grid(alpha=0.3)

# Panel 4: Feature scalings
ax4 = fig.add_subplot(2, 3, 4)
x_pos = np.arange(len(short_names))
width = 0.35
ax4.bar(x_pos - width/2, scalings[:, 0], width, label='LD1', color='#2ecc71', alpha=0.8, edgecolor='white')
ax4.bar(x_pos + width/2, scalings[:, 1], width, label='LD2', color='#3498db', alpha=0.8, edgecolor='white')
ax4.axhline(0, color='black', linewidth=0.8)
ax4.set_xticks(x_pos)
ax4.set_xticklabels(short_names, fontsize=8)
ax4.set_title('Feature Scalings per LD', fontweight='bold')
ax4.set_ylabel('Coefficient')
ax4.legend(fontsize=8)
ax4.grid(axis='y', alpha=0.3)

# Panel 5: Explained variance LDA
ax5 = fig.add_subplot(2, 3, 5)
ax5.bar(['LD1', 'LD2'], lda.explained_variance_ratio_ * 100,
        color=['#2ecc71', '#27ae60'], edgecolor='white', alpha=0.85)
ax5.set_title('LDA Explained Variance', fontweight='bold')
ax5.set_ylabel('Variance Explained (%)')
ax5.set_ylim(0, 110)
for i, v in enumerate(lda.explained_variance_ratio_ * 100):
    ax5.text(i, v + 1, f'{v:.1f}%', ha='center', fontweight='bold', fontsize=10)
ax5.grid(axis='y', alpha=0.3)

# Panel 6: CV accuracy
ax6 = fig.add_subplot(2, 3, 6)
ax6.bar(range(1, 6), cv_scores * 100, color='#9b59b6', alpha=0.8, edgecolor='white')
ax6.axhline(cv_scores.mean() * 100, color='#e74c3c', linewidth=2.5,
            linestyle='--', label=f'Mean: {cv_scores.mean()*100:.1f}%')
ax6.set_title('LDA Classifier CV Accuracy', fontweight='bold')
ax6.set_xlabel('Fold')
ax6.set_ylabel('Accuracy (%)')
ax6.set_ylim(90, 102)
ax6.set_xticks(range(1, 6))
ax6.legend(fontsize=8)
ax6.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

---
## 📚 Step 13: Conclusions & Key Takeaways

### What We Did
1. Loaded the 4D Iris dataset (150 samples, 4 features, 3 classes)
2. Scaled features with StandardScaler
3. Applied LDA to reduce 4D → 2D (maximum possible for 3 classes)
4. Visualized class separation in 1D and 2D projections
5. Inspected feature contributions (scalings)
6. Compared LDA vs PCA — both visually and with a separation score
7. Used LDA as a direct classifier (cross-validation accuracy)
8. Visualized LDA decision regions

---

### 🏆 Key Takeaways

| Concept | Takeaway |
|---|---|
| **Supervised** | LDA uses class labels — PCA does not |
| **Max components** | Always ≤ n_classes − 1 |
| **Dual use** | LDA can reduce dimensions AND classify |
| **Assumes Gaussian** | Works best when each class is normally distributed |
| **Linear only** | Fails on circular/ring-shaped class boundaries |
| **Scale first** | Always standardize features before applying LDA |

---

### 🔗 What's Next?
- 🔹 **QDA (Quadratic Discriminant Analysis)** — relaxes equal-covariance assumption
- 🔹 **Kernel LDA** — non-linear extension (like kernel PCA)
- 🔹 **LDA + SVM pipeline** — use LDA for reduction, then SVM for classification
- 🔹 **Multi-class LDA** — explore when n_classes > 3

---

### 🧠 Interview Questions — Quick Answers

**Q1: PCA vs LDA?**  
PCA maximizes total variance (unsupervised). LDA maximizes class separation (supervised).

**Q2: Max components for 4-class problem?**  
n_classes − 1 = 3 components maximum.

**Q3: Main assumptions of LDA?**  
Gaussian distributions per class, equal covariance matrices, no multicollinearity.

**Q4: Can LDA classify directly?**  
Yes — it assigns new points to the nearest class centroid in the projected space.

**Q5: When to prefer LDA over PCA?**  
When class labels are available and you want better downstream classification accuracy.